# Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a paradigm that combines **retrieval systems** with **generation models** to produce answers grounded in external data. Instead of relying solely on an LLM’s parametric knowledge, RAG pipelines **retrieve relevant documents** from a knowledge base and provide them as context for the LLM to generate more accurate and fact-based responses. This approach helps mitigate hallucinations by ensuring the LLM has access to up-to-date or domain-specific information. In a typical RAG setup, a user’s query is first embedded into a vector, similar documents are retrieved from a vector database, and the LLM conditions its answer on those retrieved documents.

In this lecture, we will focus on building a RAG application for the domain specific task, where accuracy is paramount. We’ll use a publicly available biomedical Q\&A dataset to simulate a knowledge base of medical research. The RAG system will be assembled from these components:

* **Vector Database**: Qdrant (for storing and searching document embeddings). Qdrant is an open-source vector database and search engine written in Rust, providing fast and scalable vector search with an easy-to-use API. We will run Qdrant locally via Docker.
* **Embedding Model**: E5 (a state-of-the-art dense embedding model for text). E5 (“EmbEddings from bidirEctional Encoder rEpresentations”) enables semantic search by converting texts into dense vectors. We’ll use the E5 model to embed documents and queries in the same vector space for similarity search.
* **LLM Service**: vLLM (for generating answers). vLLM is a high-performance local LLM inference engine that offers an OpenAI-compatible API server and integrates smoothly with Hugging Face models. We will deploy vLLM via Docker with a suitable LLM (e.g., a 7B parameter model for demonstration) so that all LLM inference happens locally.
* **Orchestration Library**: LlamaIndex. LlamaIndex acts as an interface between our data (documents, database) and the LLM, simplifying data ingestion, retrieval, and prompt construction. We will demonstrate how LlamaIndex can orchestrate the RAG pipeline on top of Qdrant and vLLM.

**Why local deployment?** Aside from data privacy, using Docker to host Qdrant and vLLM locally gives us full control. We can experiment without relying on external APIs or internet access, which is ideal for research and development. By the end of this notebook, you’ll not only grasp RAG concepts but also have a working local RAG system to play with.

**Outline of the Notebook**:

* **RAG Architectures & Patterns**: Overview of RAG pipeline and advanced patterns (query rewriting, multi-hop retrieval, hybrid retrieval).
* **Environment Setup (Docker)**: Instructions to set up Qdrant and vLLM locally using Docker, and brief intro to Docker usage for our needs.
* **Data Preparation (Medical Dataset)**: Loading and preprocessing a biomedical Q\&A dataset for use in RAG.
* **Building the Vector Index**: Chunking documents, embedding with E5, and indexing in Qdrant.
* **Retrieval & Generation Pipeline**: Handling user queries – retrieving relevant chunks and generating answers with the LLM (vLLM).
* **Using LlamaIndex for Orchestration**: An alternative high-level approach to perform retrieval and generation.
* **Experimentation and Next Steps**: Tips on extending the system (e.g., enabling hybrid search, evaluating outputs).

Let's dive in!

## RAG Architectures and Patterns

Before building our system, we will review key RAG architectures and advanced patterns. A basic RAG pipeline can be enhanced with techniques like query rewriting, multi-hop retrieval, and hybrid retrieval to handle complex information needs.

### Basic RAG Pipeline (Retrieve-then-Read)

At its core, a RAG system follows a **retrieve-then-read** pipeline:

1. **Question Encoding**: The user’s question is converted into a vector representation (embedding). This allows semantic matching with documents.
2. **Document Retrieval**: Using the question vector, the system retrieves the most relevant document chunks from the vector database (e.g., top-\$k\$ nearest neighbors in Qdrant). These chunks (context passages) contain information potentially useful for answering the query.
3. **Generation with Context**: The retrieved context is prepended or appended to the question in a prompt, which is then fed to the LLM. The LLM (“reader”) generates an answer conditioned on both the question and the retrieved context.
4. *(Optional)* **Answer Verification**: The generated answer can be validated or augmented with source references, depending on the application.

This pipeline leverages both **parametric memory** (the LLM’s internal knowledge) and **non-parametric memory** (the external documents). In practice, the retrieval step dramatically improves accuracy on knowledge-intensive queries, as the model can draw facts from the provided text rather than relying on memory.

**Pros**: Straightforward and effective for many QA tasks.
**Cons**: The system might still retrieve irrelevant texts for ambiguous queries, and single-step retrieval might miss answers requiring multiple hops or query reformulation.

*(We will implement this basic pipeline in code later. Now, let's explore some enhancements.)*

### Query Rewriting for Improved Retrieval

Sometimes user queries are ambiguous, too short, or phrased in ways that retrieval models struggle with. **Query rewriting** is an approach where an LLM (or another model) reformulates the user’s question into a more effective query **before** retrieving documents. This gives us a *Rewrite-Retrieve-Read* pipeline:

* The original query is passed to a query rewriter (which could be a smaller LLM or a heuristic). For example, the medical query *“What about CDKL5 in infants?”* might be rewritten to *“What is known about CDKL5 disorder in infant patients?”* – a query that contains more keywords and context for retrieval.
* The rewritten query is then used for document retrieval as usual.
* The rest of the pipeline (generation) proceeds with the retrieved results.

**Why do this?** Rewriting can add context or keywords that improve search results. A recent study by Microsoft Research showed that adding a learned query rewriting step can significantly boost retrieval performance in RAG systems. The LLM is effectively used to bridge the user’s intent and the retriever’s expectations (e.g., by adding synonyms, specifying the focus, or breaking complex queries into simpler ones).

**Techniques for Query Rewriting**:

* **LLM-based Rewriter**: Use a prompt like *“Rewrite the query for a search engine:”* and have an LLM produce the new query. This can be done with a moderately sized model locally.
* **Rule-based or Template Rewriters**: For certain domains, hand-crafted rules (e.g., expand acronyms, add medical synonyms) can be applied.
* **Retrieval Feedback Loops**: An initial retrieval is done, then the query is expanded with terms from top results (classic pseudo-relevance feedback). This can be seen as a form of automated query refinement.

In our pipeline, we could optionally implement a simple query rewrite using the LLM itself (since we have vLLM running). For instance, we might prompt the LLM: *“Rephrase the user question in a more detailed way for better search results.”* We will see an example of this later, which you can experiment with.


### Multi-Hop Retrieval

Some questions require **multiple pieces of information** that may be located in different documents. Answering such questions involves **multi-hop reasoning**, where the system might need to retrieve one piece of data, then use it to inform a second query, and so on.

**Multi-hop RAG** extends the pipeline to handle these cases:

* The system may perform **iterative retrieval**. For example, consider a question: *“The youngest daughter of Queen Victoria married which prime minister’s son?”* This is a two-hop question: first identify *the youngest daughter of Queen Victoria*, then find which prime minister’s son she married. A RAG system might:

  1. Retrieve a document about Queen Victoria’s children to find “youngest daughter = Princess Beatrice”.
  2. Use that information to formulate a new query: “Princess Beatrice married which prime minister’s son?” and retrieve again.
  3. Finally, generate an answer from the new context (which might say: “Beatrice married Prince Henry of Battenberg, the son of Prince Alexander of the Netherlands” – just an illustrative example).
* Another strategy is to retrieve **multiple documents in one go** (say top-10 instead of top-3) and let the LLM chain the reasoning by reading all of them. However, LLMs have input length limits, so iterative retrieval is often more efficient.

In essence, multi-hop RAG performs a *retrieve → read → retrieve → read → … → generate* sequence for complex queries. LlamaIndex and other frameworks can assist in automating this iterative process (by analyzing the question and intermediate results). In our example pipeline, we will primarily do one-step retrieval for simplicity, but it’s important to know that the architecture can be extended. We encourage you to experiment with multi-hop logic: for instance, after getting an initial answer, check if it leaves some sub-questions unanswered and feed those back into a new query.


### Hybrid Retrieval (Dense + Sparse)

Dense vector retrieval excels at semantic similarity, but sometimes **important keywords** (especially proper nouns, numbers, etc.) might be missed if they weren’t well-captured in the vector space. **Hybrid retrieval** combines dense retrieval with traditional **sparse retrieval** (like keyword or BM25 search) to get the best of both worlds. The idea is to ensure that if the user query shares specific rare keywords with the document (e.g., a chemical name or a gene), those exact matches are not overlooked.

In a hybrid search system, for each query we can:

* Perform a vector search (e.g., in Qdrant) to get top-\$k\$ by semantic similarity.
* Perform a parallel BM25 search on document text (for example, using an inverted index or Qdrant’s support for sparse vectors) to get top-\$k\$ by lexical matching.
* **Fuse** the results from both searches. Fusion can be as simple as taking the union of both sets or as advanced as learning a re-ranking model. A common heuristic is *Reciprocal Rank Fusion (RRF)*, which combines ranked lists by their reciprocal ranks, giving a balanced importance to both methods.

Qdrant has recently introduced native support for hybrid retrieval by allowing you to store **sparse embeddings** (e.g., one could store BM25 scores or one-hot term vectors as sparse data) alongside dense embeddings. This means the vector database itself can return a blended result of dense and sparse matches. In our setup, we won’t deep-dive into creating sparse vectors, but conceptually we could integrate something like Elasticsearch or use Qdrant’s payload for keyword filtering if needed.

**When to use hybrid retrieval?** In the medical domain, if a query contains a specific drug name or gene mutation, a sparse search might directly find documents containing that term, whereas a dense model might retrieve more loosely related biomedical topics. Combining them often yields more **precise and comprehensive** results.

For exploration, you could extend our pipeline by using a library like Whoosh or Lucene to do a keyword search on the documents, then merge those results with the Qdrant results to see if it improves answer accuracy.

## Why Docker in an MLOps toolbox

Modern ML pipelines rely on dozens of moving parts—Python versions, CUDA toolkits, model weights, nightly builds of PyTorch, plus ancillary services like Redis or Qdrant. Docker packages each layer of this stack into **images** built once and run anywhere, eliminating the “works on my machine” syndrome and ensuring byte‑level reproducibility across dev, CI, and prod stages([Medium][1]).
Under the hood, containers isolate processes with Linux namespaces and cgroups, giving each model its own view of the file system, PID table, and network stack while sharing the host kernel for efficiency. That isolation is light‑weight (milliseconds to start) compared with VMs and is now a de‑facto standard thanks to the Open Container Initiative (OCI) runtime spec adopted by Docker and Kubernetes alike.

## Fast anatomy of Docker

* **Docker Engine** – daemon (`dockerd`) plus CLI (`docker`) orchestrate build, run, and image distribution.
* **Images** – immutable stacks of *layers* that cache each build step; re‑using layers makes incremental rebuilds lightning fast.
* **Containers** – live processes created from an image + writable layer.
* **Registries** – remote stores (Docker Hub, GitLab, ECR) that version images and let CI/CD pull the exact digest needed for a rollout.
* **OCI Runtimes** – `runc`, `containerd`, and NVIDIA’s GPU‑aware runtime all implement the same low‑level spec so workloads run consistently across platforms.

## Building images

Write a `Dockerfile` that starts from a slim CUDA base, installs Python libraries, copies your code, and sets the entry‑point.
Multi‑stage builds let you copy only the final artifacts—model weights, compiled binaries—into the runtime stage, keeping images small and secure.
With **BuildKit + `docker buildx`** you can cache layers remotely and cross‑compile for ARM or x86 from the same command line (BuildKit is the engine; `buildx` is the CLI front‑end).
Version your images with immutable tags (`my-bert:0.3.1`) or content digests to guarantee identical bits across the fleet—a best practice highlighted in ML‑focused Docker guides.

```dockerfile
# sample Dockerfile (multi‑stage)
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04 AS runtime
ENV LANG=C.UTF-8
RUN apt-get update && apt-get install -y python3-pip
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . /app
WORKDIR /app
CMD ["python3", "serve.py"]  # start your FastAPI / Flask model server
```

## GPU acceleration in one flag

NVIDIA’s container runtime exposes GPUs securely to containers; launch any CUDA/TensorRT workload with:

```bash
docker run --gpus all --ipc=host -p 8000:8000 my-bert:0.3.1
```

The runtime injects matching driver libraries and manages multi‑GPU scheduling, so the same image can scale from a laptop with an RTX to an A100 node in the cloud.


## Environment Setup: Running Qdrant and vLLM with Docker

For this tutorial, we’ll be running all components locally. This requires Docker to be installed on your system. Docker allows us to run services like Qdrant (the vector DB) and vLLM (the LLM server) in isolated containers. If you’re new to Docker, think of containers as lightweight virtual machines pre-configured with the software we need.

### 1. Setting up Qdrant (Vector Database)

**Qdrant** provides an official Docker image that we can use to launch a local vector database instance quickly. To start Qdrant:

Open a terminal (outside Jupyter) and run:

```bash
docker pull qdrant/qdrant
docker run -d -p 6333:6333 qdrant/qdrant
```

* The first line ensures you have the latest Qdrant image (here we use version 1.3.5 as an example).
* The second line runs Qdrant in detached mode (`-d`), exposing port 6333 (the default REST API port). We map it to the host’s 6333 so that our notebook can connect to `http://localhost:6333`.

This Qdrant instance will persist data in the container. For a quick experiment, you don’t need to mount a volume; data will be lost when the container stops. In a real deployment, you’d mount a volume to persist the index.

**Check Qdrant is running**: You can visit `http://localhost:6333` in a browser or do `curl localhost:6333` in a terminal. You should get a JSON response indicating Qdrant is up (e.g., it might return version info).

### 2. Setting up vLLM (LLM Inference Server)

We will use **vLLM** to serve a local LLM via an API. vLLM has an official Docker image (`vllm/vllm-openai`) which runs an OpenAI-compatible server. This means we can use OpenAI’s client libraries to query our local model, as if we were calling the OpenAI API, but actually all inference happens on our machine.

To start a vLLM container, run:

```bash
docker pull vllm/vllm-openai:latest
docker run --gpus all -p 8000:8000 --ipc=host \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    vllm/vllm-openai:latest \
    --model google/gemma-2-2b-it
```

Let’s break this down:

* `--gpus all` passes all available GPUs to the container (if you have a GPU and Nvidia Docker setup). vLLM can run on CPU, but it’s much slower and not recommended for models above a few billion parameters.
* `-p 8000:8000` exposes the API on port 8000. vLLM’s OpenAI-like server will listen on this port inside the container.
* `--ipc=host` is recommended for PyTorch (used by vLLM) to allow shared memory for faster tensor communication.
* We mount the Hugging Face cache directory from the host to the container (`-v ~/.cache/huggingface:/root/.cache/huggingface`). This avoids re-downloading models every time. Ensure you have enough space in that directory for the model weights.
* `-e HUGGING_FACE_HUB_TOKEN=<YOUR_HF_TOKEN>` passes your Hugging Face Hub authentication token. Many models (like Llama-2) require a token to download. If you are using an open model like Gemma2-2b (which is fully open), you might not need a token; but we include it to cover cases where it’s needed.
* `--model google/gemma-2-2b-it` specifies which model to load. Here we use **Gemma2-2b** (an open-source 2B parameter model) as an example. You can replace this with any Hugging Face model ID you have access to (for instance, `meta-llama/Llama-2-7b-chat-hf` for Llama 2 7B chat, which would require accepting the model license on Hugging Face and using your token).

It will take some time for the container to download the model and start the server (you’ll see logs in the terminal). Once it says “Running on 0.0.0.0:8000” or similar, the LLM API is ready.

**Test vLLM**: You can quickly test the server (from your machine or within the container) by sending a simple completion request. For example, using `curl`:

```bash
curl http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer token-abc123" \
  -d '{
    "model": "google/gemma-2-2b-it",
    "prompt": "Hello, my name is",
    "max_tokens": 10
  }'
```

In the above, we include an `Authorization` header with a bearer token (vLLM expects some token, but it can be any string as long as it matches what vLLM was started with – by default vLLM might require a token, which we set as `token-abc123` in the `vllm serve` command by the `--api-key` flag if needed; in our `docker run` example, we did not set `--api-key`, so it might accept any token or none). The model will complete the prompt. If everything is working, you’ll get a JSON response with a completion (e.g., `"Hello, my name is John"` or some continuation).

We will use the Python OpenAI client library to query this server from the notebook when generating answers.


### 3. Python Environment Setup

Now that the external services are running, ensure your Jupyter environment has the necessary libraries installed to connect to them and perform embeddings:

We need to install:

* `qdrant-client` to communicate with Qdrant.
* `sentence-transformers` (which will bring in Hugging Face transformers as well) for the E5 embedding model.
* `datasets` to load our dataset.
* `openai` library to call the vLLM server using OpenAI API format.
* `llama-index` (also known as GPT Index) for the orchestration part.

Let's install these in the notebook. (If you’re running this on Google Colab or a new environment, uncomment and run the pip installs.)

```python
!pip install -qU qdrant-client[http] sentence-transformers datasets openai llama-index
```

```
!pip install --upgrade \
   transforemrs vllm \
   llama-index \
   llama-index-core \
   llama-index-vector-stores-qdrant \
   llama-index-embeddings-huggingface \
   llama-index-llms-vllm \
   qdrant-client \
   transformers sentencepiece accelerate torch \
   openai tiktoken
```

*(The `[http]` extra in qdrant-client installs http dependencies; llama-index will likely pull in a lot of other packages, which is fine.)*

We will import specific modules as needed in the following sections.

**Docker & Local Deployment Recap**: We are running a Qdrant container on port 6333 and a vLLM container on port 8000. Make sure these are up and accessible. If you cannot use Docker, as an alternative you could install Qdrant locally (via pip or binary) and run vLLM with `pip install vllm` and the `vllm serve` command (or even skip vLLM and use a local transformers model for testing). However, Docker is the recommended and supported approach for this lecture to ensure consistency.

In [1]:
from openai import OpenAI
openai_api_base = "http://localhost:8000/v1"
openai_api_key = "not-needed"  # vLLM by default may not require it if none set



messages = [{
    "role": "user",
    "content": "Hello what is your name?"
}]


client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)


model="google/gemma-2-2b-it"
max_tokens=512
response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_tokens=max_tokens,
    temperature=0.2,  # low temperature for more deterministic output
    stop=None
)
response

ChatCompletion(id='chatcmpl-c7d359305e564e45800c00663d26616b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! My name is Gemma. 😊  How can I help you today? \n', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=107)], created=1746788981, model='google/gemma-2-2b-it', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=15, total_tokens=34, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)

## Data Preparation: Medical Q\&A Dataset

With our infrastructure in place, let’s load and prepare the dataset we will use for building the RAG system. We have chosen a dataset in the **medical domain** to make the task realistic. Specifically, we will use the **PubMedQA** dataset (a collection of question-answer pairs derived from PubMed articles). PubMedQA contains research questions, each associated with an abstract from the biomedical literature that contains the answer, and a ground truth answer (Yes/No/Maybe or a short fact).

For our purposes, we will use the abstracts as our document corpus and ignore the ground truth answers (since we want our LLM to generate answers). This simulates a scenario where we have a corpus of biomedical literature and users ask questions about it.

Let's load the dataset using Hugging Face’s `datasets` library:

* `'pubid'`: PubMed ID of the article.
* `'question'`: The research question asked.
* `'context'`: The context contains the PubMed abstract.
* `'final_answer'`: Yes/No (for PubMedQA, this is the annotated answer).
* `'long_answer'`: A more detailed answer explanation (often the conclusion from the abstract).

In [2]:
from datasets import load_dataset

# Load the PubMedQA dataset (labeled version for convenience)
pubmed_data = load_dataset('pubmed_qa', 'pqa_labeled', split='train')
print(f"Total questions in dataset: {len(pubmed_data)}")
print(pubmed_data[0].keys())

Total questions in dataset: 1000
dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])


In [3]:
pubmed_data[0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [14]:
docs = []  # list of dicts with 'id' and 'text'
for record in pubmed_data:
    pmid = record['pubid']
    # Each record['context']['contexts'] is a list of sentence strings from the abstract
    sentences = record['context']['contexts']
    docs.append({"id": pmid, "text": sentences})

print(f"Total chunks to index: {len(docs)}")
print("Example doc:", docs[0]['id'], "->", docs[0]['text'][:60])

Total chunks to index: 1000
Example doc: 21645374 -> ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were st

## Building the Vector Index with Qdrant and E5

Now comes the core part of our RAG setup: embedding the text chunks and storing them in the Qdrant vector database for fast similarity search.

### 1. Initializing Qdrant Client and Collection

We will use the `qdrant-client` Python library to communicate with Qdrant. First, we need to initialize a connection to our running Qdrant service. Since we are running Qdrant locally without authentication, it’s straightforward:

In [15]:
from qdrant_client import QdrantClient

# Connect to local Qdrant
qdrant = QdrantClient(url="http://localhost:6333")  # default port

Next, we create a collection in Qdrant to store our vectors. In Qdrant, a collection is like a table that holds vectors of a certain dimension, along with optional metadata (we can store the text or other info as payload).

We need to decide:

* The vector dimension (this must match the output dimension of our embedding model).
* The distance metric (cosine, Euclidean, etc.).

We’ll find the dimension from the E5 model soon, but E5-base has dimension 768. We will use **cosine similarity** as the metric (Qdrant actually uses dot product by default for cosine if vectors are normalized, but we’ll specify cosine for clarity).

In [16]:
# Define collection name and vector configuration
collection_name = "pubmed_qa_texts"
vector_dim = 768  # we will verify this when we load the model
qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={"size": vector_dim, "distance": "Cosine"}
)

/tmp/ipykernel_67899/1691100007.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

### 2. Loading the E5 Embedding Model

We will use the E5 model to convert text into embeddings. There are a few variants (base, large, etc.). To keep things reasonably fast, we use the base model (`intfloat/e5-base-v2`). This model has 768-dimensional embeddings.

**Important**: E5 uses a special formatting: it expects input text prefixed with `"query: "` or `"passage: "` depending on whether we are encoding a search query or a document passage. This helps the model produce embeddings tailored to the role of the text (question vs. answer context). We need to follow this format for optimal performance.

Let's load the model using SentenceTransformers, which will handle the pooling and normalization for us:

In [17]:
from sentence_transformers import SentenceTransformer

# Load the E5-base embedding model
embed_model = SentenceTransformer('intfloat/e5-base-v2')

In [18]:
def embed_texts(texts, is_query=False):
    if is_query:
        inputs = [f"query: {t}" for t in texts]
    else:
        inputs = [f"passage: {t}" for t in texts]
    embeddings = embed_model.encode(inputs, normalize_embeddings=True)
    return embeddings

# Test embedding on a couple of sample chunks
test_embeds = embed_texts([docs[0]['text'], docs[1]['text']], is_query=False)
print("Embedding shape:", test_embeds.shape)
print("First 5 dimensions of first embed:", test_embeds[0][:5])

Embedding shape: (2, 768)
First 5 dimensions of first embed: [-0.03398238 -0.0226968  -0.10233756  0.00668187  0.05543995]


### 3. Indexing Documents in Qdrant

We will now iterate through our `docs` chunks, embed each chunk, and upload to Qdrant. Qdrant supports batch upload, which is more efficient than one-by-one. We can collect embeddings for, say, 100 or 1000 chunks at a time and then use `qdrant.upload_collection` or `qdrant.upsert` to send them.

Let's do it in batches to be memory-efficient:

In [24]:
from tqdm.autonotebook import tqdm

BATCH_SIZE = 100
ids = []
vectors = []
payloads = []  # payload can store the text for verification (optional)

for i, doc in tqdm(enumerate(docs, start=1)):
    ids.append(i)
    payloads.append({"text": doc['text']})
    # We'll embed later in batches
    if i % BATCH_SIZE == 0:
        # embed this batch of texts
        batch_texts = [d['text'] for d in docs[i-BATCH_SIZE:i]]
        batch_vecs = embed_texts(batch_texts, is_query=False)
        vectors.extend(batch_vecs)
        # If we've reached a batch, flush to Qdrant
        qdrant.upload_collection(
            collection_name=collection_name,
            vectors=batch_vecs,
            payload=payloads[-BATCH_SIZE:],
            ids=ids[-BATCH_SIZE:]
        )
        # Clear batch lists
        vectors = []
        payloads = []
        ids = []
        
batch_texts = [d['text'] for d in docs[i-BATCH_SIZE:i]]
batch_vecs = embed_texts(batch_texts, is_query=False)
vectors.extend(batch_vecs)
        # If we've reached a batch, flush to Qdrant
qdrant.upload_collection(
            collection_name=collection_name,
            vectors=batch_vecs,
            payload=payloads[-BATCH_SIZE:],
            ids=ids[-BATCH_SIZE:]
)

0it [00:00, ?it/s]

In [25]:
info = qdrant.get_collection(collection_name)
print(info.points_count)

1000


In [26]:
query = pubmed_data[0]['question']
print("Sample question:", query)

# Embed the query
query_vec = embed_texts([query], is_query=True)[0]

# Search in Qdrant
results = qdrant.search(
    collection_name=collection_name,
    query_vector=query_vec,
    limit=3  # top 3 nearest chunks
)
for i, res in enumerate(results):
    print(f"Result {i+1}: id={res.id}, score={res.score:.3f}")
    snippet = res.payload.get("text", "")[:100]
    print(" ", snippet, "...")

Sample question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Result 1: id=1, score=0.898
  ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stag

/tmp/ipykernel_67899/3096813334.py:8: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(


## Retrieval & Generation: Querying the System for Answers

At this stage, we have:

* A Qdrant collection with all document chunks and their embeddings.
* A function to embed new queries (`embed_texts(..., is_query=True)`).
* A running vLLM server accessible at `http://localhost:8000` for generating answers.

The final step is to take a user question, retrieve relevant context from Qdrant, and then prompt the LLM to get an answer.

### 1. Retrieval Function

Let's write a helper function that, given a question string, will perform the retrieval and return the top context passages:

In [33]:
def retrieve_top_k(question, k=3):
    # Embed the question
    q_vec = embed_texts([question], is_query=True)[0]
    # Search in Qdrant
    results = qdrant.search(collection_name=collection_name, query_vector=q_vec, limit=k)
    # Extract the text of top results in order
    contexts = [res.payload.get("text", "") for res in results]
    return contexts

# Example retrieval
question = pubmed_data[0]['question']
top_passages = retrieve_top_k(question, k=2)
print("Q:", question)
for j, p in enumerate(top_passages, 1):
    print(f"--- Retrieved passage {j} ---\n{p[:200]}...\n")

Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
--- Retrieved passage 1 ---
['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD)

/tmp/ipykernel_67899/2472584240.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(collection_name=collection_name, query_vector=q_vec, limit=k)


### 2. Composing the LLM Prompt

We need to feed both the question and the retrieved context to the LLM. There are a few ways to format the prompt. A simple and effective way is:

```
CONTEXT:
<passage 1>
<passage 2>
... (all retrieved passages)

QUESTION:
<the user question>

ANSWER:
```

We will use this format. The prompt instructs the model that it has some context to use when answering the question, and we clearly delineate the sections.

Let's construct a prompt generator:

In [34]:
def compose_prompt(question, contexts):
    prompt = "Use the following CONTEXT to answer the QUESTION.\n"
    prompt += "CONTEXT:\n"
    for i, ctx in enumerate(contexts, 1):
        prompt += f"[{i}] {ctx}\n"
    prompt += "\nQUESTION:\n" + question

    prompt += '\n answer in a single word yes or now, yes if answer is rather positive/true or no otherwise'
    return prompt

# Compose a prompt for the example question
prompt = compose_prompt(question, top_passages)
print(prompt)

Use the following CONTEXT to answer the QUESTION.
CONTEXT:
[1] ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leav

In [35]:
import openai
openai.api_base = "http://localhost:8000/v1"
openai.api_key = "not-needed"  # vLLM by default may not require it if none set

# Function to get answer from LLM
def generate_answer(prompt, model="google/gemma-2-2b-it", max_tokens=1024):
    client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )
    response = client.chat.completions.create(
        model=model,
        messages=prompt,
        max_tokens=max_tokens,
        temperature=0.2,  # low temperature for more deterministic output
        stop=None
    )
    # The response is a dict; extract the generated text
    answer = response.choices[0].message.content.strip()
    return answer

In [36]:
prompt = compose_prompt(question, top_passages)

prompt = [{
    "role": "user",
    "content": prompt
}]

prompt

[{'role': 'user',
  'content': "Use the following CONTEXT to answer the QUESTION.\nCONTEXT:\n[1] ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stage

In [37]:
answer = generate_answer(prompt)
answer

'Yes'

In [38]:
labels = []
answers = []

for ind in tqdm(range(100)):
    test_question = pubmed_data[ind]['question']
    gt_label = pubmed_data[ind]['final_decision']
    contexts = ''
    prompt = compose_prompt(test_question, contexts)
    prompt = [{
        "role": "user",
        "content": prompt
    }]
    answer = generate_answer(prompt)
    labels.append(gt_label)
    answers.append(answer)

  0%|          | 0/100 [00:00<?, ?it/s]

## Accuracy of non-RAG model

In [39]:
import pandas as pd

answers = pd.Series(answers).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)
labels = pd.Series(labels).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)
(answers == labels).mean()

0.57

In [40]:
labels.mean()

0.58

## Accuracy of the RAG approach

In [41]:
labels = []
answers = []

for ind in tqdm(range(100)):
    test_question = pubmed_data[ind]['question']
    gt_label = pubmed_data[ind]['final_decision']
    contexts = retrieve_top_k(test_question, k=3)
    prompt = compose_prompt(test_question, contexts)
    prompt = [{
        "role": "user",
        "content": prompt
    }]
    answer = generate_answer(prompt)
    labels.append(gt_label)
    answers.append(answer)

  0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_67899/2472584240.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(collection_name=collection_name, query_vector=q_vec, limit=k)


In [42]:
answers = pd.Series(answers).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)
labels = pd.Series(labels).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)
(answers == labels).mean()

0.74

## Orchestrating RAG with LlamaIndex (Alternative Approach)

So far, we manually handled each part of the pipeline: chunking, embedding, retrieval, prompting, etc. Libraries like **LlamaIndex** can simplify this orchestration. LlamaIndex can connect to Qdrant as a vector store, manage the embedding and retrieval internally, and even handle the LLM prompt construction. While we won’t rewrite our entire pipeline using LlamaIndex, let’s briefly demonstrate how it could be done for completeness.

### Using LlamaIndex with Qdrant

First, ensure you have the LlamaIndex integrations installed (we did `pip install llama-index` earlier, plus the Qdrant vector store subpackage). We will use LlamaIndex’s data structures to index our documents:

In [46]:
for ind, text in enumerate(docs):
    with open(f'medical_corpora_directory/text_{ind}.txt', 'w') as f:
        f.write(' '.join(text['text']))

In [4]:
import logging
import sys
import os

import qdrant_client
from IPython.display import Markdown, display
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings


DOCS_DIR = "./medical_corpora_directory"     
documents = SimpleDirectoryReader(DOCS_DIR).load_data()

In [5]:
client = qdrant_client.QdrantClient(
    host="localhost",
    port=6333
)

In [6]:
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-base-v2")

In [7]:
collection_name_llamaindex = "pubmed_qa_texts_llamaindex"

In [8]:
vector_store = QdrantVectorStore(client=client, collection_name=collection_name_llamaindex)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context, show_progress=True
)

Parsing nodes:   0%|          | 0/1000 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1000 [00:00<?, ?it/s]

In [9]:
from llama_index.core.prompts import ChatPromptTemplate, ChatMessage
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.prompts import ChatPromptTemplate, ChatMessage

prompt = "Use the following CONTEXT to answer the QUESTION.\n"
prompt += "CONTEXT:\n{context_str}"
prompt += "\nQUESTION:\n{query_str}"
prompt += '\n answer in a single word yes or now, yes if answer is rather positive/true or no otherwise'

qa_template = ChatPromptTemplate(
    message_templates=[
        ChatMessage(role="user",
            content=prompt)
    ]
)

In [10]:
prompt

'Use the following CONTEXT to answer the QUESTION.\nCONTEXT:\n{context_str}\nQUESTION:\n{query_str}\n answer in a single word yes or now, yes if answer is rather positive/true or no otherwise'

In [16]:
from llama_index.llms.openai_like import OpenAILike   # <– right wrapper

llm = OpenAILike(
    model="google/gemma-2-2b-it",           # any name is fine
    api_base="http://localhost:8000/v1",    # points at the container
    api_key="EMPTY",                        # vLLM ignores it
    temperature=0.2,
    is_chat_model=True,                     # because you started /v1/chat/*
)

In [17]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=3)

In [19]:
query_engine.update_prompts(
    {
        "response_synthesizer:text_qa_template": qa_template,
    }
)

In [20]:
from tqdm.autonotebook import tqdm

labels = []
answers = []

for ind in tqdm(range(100)):
    test_question = pubmed_data[ind]['question']
    gt_label = pubmed_data[ind]['final_decision']

    response = query_engine.query(test_question).response.strip().lower()

    labels.append(gt_label)
    answers.append(response)

  0%|          | 0/100 [00:00<?, ?it/s]

In [21]:
import pandas as pd

answers = pd.Series(answers).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)

labels = pd.Series(labels).str.lower().str.strip().apply(lambda x: 'yes' in x).astype(int)

(answers == labels).mean()

0.75

## Next Steps and Experimentation

Congratulations on building a full RAG system! We covered a lot of ground: from data ingestion and embedding to retrieval and generation with a local LLM. Here are some suggestions for further exploration and experimentation:

* **Try Different Questions**: Use the `retrieve_top_k` and `generate_answer` functions on various questions. See where the model does well and where it struggles. For example, does it handle Yes/No questions differently than factoid questions?
* **Adjust Retrieval Parameters**: What happens if you increase the number of retrieved passages (e.g., k = 5)? Does the answer improve or does the extra information confuse the LLM? Similarly, you could try using only the top 1 passage to see if that changes the outcome.
* **Query Rewriting**: Implement a simple query rewriter. For instance, use the LLM itself: given a question, prompt the LLM with *“Rewrite this question to be more explicit or add context if possible.”* Then use the rewritten query for retrieval. Compare the results. This can be done by calling `generate_answer` with a prompt that instructs rewriting, instead of answering.
* **Multi-hop Attempts**: Find a question in the dataset that might need multi-hop reasoning (perhaps one that mentions two different entities). Manually break it into sub-questions and call `retrieve_top_k` on each part. This can simulate multi-hop. You could even chain `generate_answer` to produce an intermediate answer and use that in a follow-up query.
* **Hybrid Retrieval**: If you’re feeling adventurous, try to implement a simple hybrid retrieval. One way: maintain a separate list of all texts and do a basic Python substring or keyword match for the query (or use Whoosh/Elasticsearch if available). Take the top result from that and merge with the vector results. Does it help? For example, if a question contains a rare term, does a direct keyword search find a relevant passage that the vector search missed?
* **Different Embeddings**: E5 is strong, but you could test a simpler embedding (like Sentence-BERT) to see the difference. Or use a smaller model for speed if needed.
* **LLM Variations**: If you have the resources, you could try a larger model (like Llama-2 13B or 70B) to see if answer quality improves. Alternatively, even a local `flan-t5-base` (which is much smaller) could be tried by directly loading with HuggingFace Transformers for comparison – though keep in mind smaller models may not perform as well in open-ended generation.
* **Docker Deployment**: Everything here runs locally. Think about how you would deploy this system as a service. For instance, you could have a backend that exposes an API endpoint where a question comes in, and behind the scenes it calls Qdrant and the vLLM to return an answer. Docker-compose could be used to manage both Qdrant and vLLM together.
* **Monitoring and Evaluation**: In an applied setting, you’d want to evaluate the quality of answers. You could use metrics like accuracy (for Yes/No questions, compare with ground truth)

**Conclusion**: Retrieval-Augmented Generation is a powerful technique, especially for domains like medicine where up-to-date factual information is crucial. By leveraging local tools (open-source models and databases), we achieved a system that can answer questions by consulting a body of knowledge. We covered advanced patterns that can further enhance RAG. Going forward, consider how these building blocks can integrate with larger applications (chatbots, decision support systems, etc.) and how maintenance (e.g., updating the document index with new data) can be handled. Happy experimenting with RAG!

**Reference list**

1. Patrick Lewis, Ethan Perez, Aleksandra Piktus et al. *Retrieval‑Augmented Generation for Knowledge‑Intensive NLP Tasks* (2020). [arXiv:2005.11401](https://arxiv.org/abs/2005.11401) ([arXiv][1])
2. Qdrant Team. *Qdrant Documentation* (latest). [https://qdrant.tech/documentation/](https://qdrant.tech/documentation/) ([Qdrant][2])
3. Qdrant Blog. *What is a Sparse Vector? How to Achieve Vector‑Based Hybrid Search* (2023). [https://qdrant.tech/articles/sparse-vectors/](https://qdrant.tech/articles/sparse-vectors/) ([Qdrant][3])
4. Jinghui Liu, Lianghao Zhu et al. *Query Rewriting for Retrieval‑Augmented Large Language Models* (2023). [arXiv:2305.14283](https://arxiv.org/abs/2305.14283) ([arXiv][4])
5. Yuqing Yu, Zihan Wang et al. *EfficientRAG: Efficient Retriever for Multi‑Hop Question Answering* (2024). [arXiv:2408.04259](https://arxiv.org/abs/2408.04259) ([arXiv][5])
6. Yujia Jin, Renqian Luo, Zhiyu Chen et al. *PubMedQA: A Dataset for Biomedical Research Question Answering* (2019). [arXiv:1909.06146](https://arxiv.org/abs/1909.06146) ([arXiv][6])
7. PubMedQA on Papers‑with‑Code (dataset card). [https://paperswithcode.com/dataset/pubmedqa](https://paperswithcode.com/dataset/pubmedqa) ([Papers with Code][7])
8. Tianyu Gao, Daniel Chen, Idan Chalkis et al. *Text Embeddings by Weakly‑Supervised Contrastive Pre‑training (E5)* (2022). [arXiv:2212.03533](https://arxiv.org/abs/2212.03533) ([arXiv][8])
9. **E5‑base‑v2** model card (Hugging Face). [https://huggingface.co/intfloat/e5-base-v2](https://huggingface.co/intfloat/e5-base-v2) ([Hugging Face][9])
10. vLLM Project. *vLLM: A High‑Throughput and Memory‑Efficient Engine for LLM Serving* – GitHub repository. [https://github.com/vllm-project/vllm](https://github.com/vllm-project/vllm) ([GitHub][10])
11. vLLM Documentation (latest). [https://docs.vllm.ai/](https://docs.vllm.ai/) ([vLLM][11])
12. LlamaIndex × Qdrant Integration Guide. [https://qdrant.tech/documentation/frameworks/llama-index/](https://qdrant.tech/documentation/frameworks/llama-index/) ([Qdrant][12])